## **CONTENT BASED RECOMMENDATION**

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_path = Path('../data/raw')


songs_data_path = data_path / 'Music Info.csv'
users_data_path = data_path / 'User Listening History.csv'

In [3]:
# load the songs data

df_songs = pd.read_csv(songs_data_path)
df_songs.head()

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...,09ZQ5TmUG8TSL56n0knqrj,"rock, alternative, indie, alternative_rock, in...",NaN,2004,222200,0.355,...,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...,06UfBBDISthj1ZJAtX4xjj,"rock, alternative, indie, pop, alternative_roc...",NaN,2006,258613,0.409,...,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,0keNu0t0tqsWtExGM3nT1D,"rock, alternative, alternative_rock, 90s, grunge",RnB,1991,218920,0.508,...,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...,0ancVQ9wEcHVd0RrGICTE4,"rock, alternative, indie, alternative_rock, in...",NaN,2004,237026,0.279,...,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,01QoK9DA7VTeTSE3MNzp4I,"rock, alternative, indie, alternative_rock, in...",RnB,2008,238640,0.515,...,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


### **GETTING THE DATASET READY**

In [ ]:
## SHAPE
df_songs.shape

(50683, 21)

In [ ]:
## INFO
df_songs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50683 entries, 0 to 50682
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50683 non-null  object 
 1   name                 50683 non-null  object 
 2   artist               50683 non-null  object 
 3   spotify_preview_url  50683 non-null  object 
 4   spotify_id           50683 non-null  object 
 5   tags                 49556 non-null  object 
 6   genre                22348 non-null  object 
 7   year                 50683 non-null  int64  
 8   duration_ms          50683 non-null  int64  
 9   danceability         50683 non-null  float64
 10  energy               50683 non-null  float64
 11  key                  50683 non-null  int64  
 12  loudness             50683 non-null  float64
 13  mode                 50683 non-null  int64  
 14  speechiness          50683 non-null  float64
 15  acousticness         50683 non-null 

In [7]:
## MISSING VALUES
df_songs.isna().sum()

track_id                   0
name                       0
artist                     0
spotify_preview_url        0
spotify_id                 0
tags                    1127
genre                  28335
year                       0
duration_ms                0
danceability               0
energy                     0
key                        0
loudness                   0
mode                       0
speechiness                0
acousticness               0
instrumentalness           0
liveness                   0
valence                    0
tempo                      0
time_signature             0
dtype: int64

In [11]:
## % OF MISSING VALUES
(
    df_songs
    .isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .head(2)
)

genre    55.906320
tags      2.223625
dtype: float64

- **LEARNING**
> 56% data is not available for Genre Column so we should drop it and for tags we can fill missing values as No tag.

In [ ]:
## duplicates in the data based on spotify_id

(
    df_songs
    .duplicated(subset=['spotify_id'])
    .sum()
)

np.int64(9)

- **Learning**
> During EDA we also Found that 9 rows are duplicated based on the spotify_id, year, duration_ms columns


In [14]:
## DROPPING THOSE DUPLICATED ROWS
df_songs.drop_duplicates(subset=['spotify_id','year','duration_ms'],inplace=True)

In [15]:
# VERIFY 

(
    df_songs
    .duplicated(subset=["spotify_id","year","duration_ms"])
    .sum()
)

np.int64(0)

In [16]:
## RESETTING THE INDEX AFTER DROPPING THE DUPLICATES
df_songs.reset_index(drop=True,inplace=True)

- **NOTE**:-
> As, Here we are doing the content based filtering so that means we will gonna use the cosine similarity to find the similarity. But like some of the features are not that helpful with cosine similarity because they have very unique data for example track_id mostly different for each song so cosine sim will not find anything helpfule their. So we will drop those column

In [19]:
## REMOVING THE UNNECESSARY COLUMNS

cols_to_drop = ['track_id','name','spotify_preview_url','spotify_id','genre']

df_content_filter = df_songs.drop(columns=cols_to_drop)

df_content_filter.head()

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,Nirvana,"rock, alternative, alternative_rock, 90s, grunge",1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,Franz Ferdinand,"rock, alternative, indie, alternative_rock, in...",2004,237026,0.279,0.664,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,Radiohead,"rock, alternative, indie, alternative_rock, in...",2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


In [20]:
## CHECK FOR MISSING VALUES IN THE NEW DATAFRAME
df_content_filter.isna().sum() 

artist                 0
tags                1126
year                   0
duration_ms            0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
speechiness            0
acousticness           0
instrumentalness       0
liveness               0
valence                0
tempo                  0
time_signature         0
dtype: int64

In [21]:
## FILL THE TAGS COLUMN MISSING VALUS WITH STRING 'no_tags'
df_content_filter.fillna({'tags':'no_tags'}, inplace=True)

In [22]:
## CHECK THE MISSING VALUES
df_content_filter.isna().sum()

artist              0
tags                0
year                0
duration_ms         0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64

In [23]:
## ARTIST NAMES AS LOWERCASE
df_content_filter['artist'] = df_content_filter['artist'].str.lower()

In [25]:
## NUMBER OF UNIQUE YEAR VALUES
(
    df_songs
    .loc[:,'year']
    .nunique()
)

75

In [ ]:
# MIN AND MAX VALUES OF THE NUMERICAL COLUMNS
(
    df_songs
    .select_dtypes(include='number')
    .agg(['min','max'])
)

,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
min,1900,1439,0.000,0.0,0,-60.000,0,0.000,0.000,0.000,0.000,0.000,0.000,0
max,2022,3816373,0.986,1.0,11,3.642,1,0.954,0.996,0.999,0.999,0.993,238.895,5


In [ ]:
# VALUE COUNTS OF THE TAGS

(
    df_songs
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
)

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
dark_ambient      602
japanese          489
polish            411
j_pop             213
russian           127
Name: count, Length: 100, dtype: int64

In [29]:
# VALUE COUNTS OF THE TAGS THAT APPEAR AT LEAST 1000 TIMES

(
    df_songs
    .loc[:,'tags']
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .loc[lambda ser: ser >= 1000]
)

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
ska              1088
gothic_metal     1072
grindcore        1040
french           1018
nu_metal         1006
Name: count, Length: 85, dtype: int64

## **DATA ENCODING**

In [34]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from category_encoders.count import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

In [35]:
df_content_filter.shape

(50674, 16)

In [37]:
df_content_filter.head(2)

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,the killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.0,0.0971,0.240,148.114,4
1,oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.0,0.2070,0.651,174.426,4


In [38]:
## TRANSFORM THE COLUMNS USING COLUMN TRANSFORMER
frequency_encode_cols = ['year']
ohe_cols = ['artist','time_signature','key']
tfidf_col = 'tags'
standard_scale_cols = ['duration_ms','loudness','tempo']
min_max_scale_cols = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']

In [39]:
len(frequency_encode_cols + ohe_cols + standard_scale_cols + min_max_scale_cols)

14

In [43]:
## APPLY THE COLUMNTRANSFORMER
transformer = ColumnTransformer(transformers=[
    ('frequency_encode',CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
    ('ohe',OneHotEncoder(handle_unknown='ignore'),ohe_cols),
    ('tfidf',TfidfVectorizer(max_features=85),tfidf_col),
    ('standard_scale',StandardScaler(),standard_scale_cols),
    ('min_max_scale',MinMaxScaler(),min_max_scale_cols)
],remainder='passthrough',n_jobs=-1)

transformer

,transformers,"[('frequency_encode', ...), ('ohe', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,verbose,0
,cols,None
,drop_invariant,False


In [44]:
## FIT THE TRANSFORMER
transformer.fit(df_content_filter)

,transformers,"[('frequency_encode', ...), ('ohe', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,verbose,0
,cols,None
,drop_invariant,False


In [47]:
## TRANSFORM THE DATA
transformed_df = transformer.transform(df_content_filter)

In [48]:
transformed_df.shape

(50674, 8431)

In [49]:
transformed_df

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 907911 stored elements and shape (50674, 8431)>

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

In [54]:
## TRY TO FIND THE TOP 10 SIMILAR SONGS OF "WHENEVER, WHENEVER"

song_input = df_content_filter[df_songs['name'] == 'Whenever, Wherever']
song_input

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,shakira,"rock, pop, female_vocalists, singer_songwriter...",2012,196826,0.787,0.828,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [55]:
## TRANSFORM THE INPUT SONG
input_vector = transformer.transform(song_input)
input_vector.shape

(1, 8431)

In [56]:
## CALCULATE THE COSINE SIMILARITY BETWEEN THE INPUT VECTOR AND ALL THE OTHER SONGS
similarity_scores = cosine_similarity(transformed_df,input_vector)
similarity_scores.shape

(50674, 1)

In [57]:
similarity_scores

array([[0.99999914],
       [0.99999847],
       [0.99999921],
       ...,
       [0.99999877],
       [0.9999992 ],
       [0.99999891]], shape=(50674, 1))

In [ ]:
np.argsort(similarity_scores.ravel())

array([25334, 15657, 40529, ...,  6046, 12305,  1025], shape=(50674,))

In [59]:
np.argsort(similarity_scores.ravel())[-11:]

array([ 6287, 38383,  6526,  6121,  7172,  6133, 17241,  6129,  6046,
       12305,  1025])

In [61]:
top_10_songs_indexes = np.argsort(similarity_scores.ravel())[-11:][::-1]
top_10_songs_indexes

array([ 1025, 12305,  6046,  6129, 17241,  6133,  7172,  6121,  6526,
       38383,  6287])

In [ ]:
## TOP 10 SONGS NAMES SIMILAR TO "WHENEVER, WHEREVER"
top_10_songs_names = df_songs.loc[top_10_songs_indexes]

In [63]:
top_10_songs_names

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
12305,TRYFVKK128F4240FE8,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...,0HiJFRxWme9myvUiDlqQ8q,"pop, experimental, singer_songwriter, dance",NaN,2001,221240,0.887,...,1,-5.535,0,0.0431,0.14400,0.000590,0.1230,0.399,129.943,4
6046,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6129,TRBAUVN128F932FEF8,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...,095uakqDYR50Uza0mxvPWB,"pop, female_vocalists, dance, 00s",Pop,2014,211786,0.751,...,1,-5.351,0,0.0435,0.34000,0.000018,0.2550,0.886,95.045,4
17241,TRMBDIR128F4279C1F,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...,1BhxPx4evrx8X02RHGrLdi,"pop, dance, rnb, 00s",Rock,2007,182680,0.718,...,1,-3.959,0,0.0360,0.35300,0.000390,0.1020,0.805,117.067,4
6133,TRDXCSH128F92ED4A1,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...,09mkdGhqb5ySKVsnkx9hy2,"pop, female_vocalists, dance, soul, hip_hop, r...",NaN,2001,207733,0.835,...,1,-4.364,0,0.2840,0.00247,0.000000,0.1710,0.586,103.358,4
7172,TRLQBEN128F92E7415,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...,0RgHkNpqtAHGGBVro6mmqD,"pop, female_vocalists",Country,2016,188493,0.741,...,1,-5.362,0,0.1080,0.01950,0.000000,0.0822,0.735,107.960,4
6121,TRGZIMZ128F930A016,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...,0rpndqrkU9y9nckNCfjcq6,"pop, female_vocalists, dance, 80s",NaN,2009,242946,0.708,...,1,-4.736,0,0.0362,0.39200,0.000001,0.0561,0.968,99.953,4
6526,TRXHWIA128E078A9BB,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...,0BmUCeyXpTrUVahHKVFxuB,"pop, female_vocalists, dance, 80s, new_wave",NaN,1984,209573,0.665,...,1,-5.828,0,0.0278,0.25000,0.008550,0.0537,0.932,108.429,4
38383,TRWUWRZ128F42ADA4A,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...,2ObxMmMaDINr0ynkqW2BlY,"pop, female_vocalists, guitar, pop_rock",Pop,2005,242760,0.689,...,1,-7.427,1,0.0286,0.18000,0.000038,0.0844,0.548,96.098,4


In [69]:
def recommend(song_name,songs_data,transformed_data,k=10):
    ## FILTER OUT THE SONG FROM THE DATA
    song_row = songs_data[songs_data['name'] == song_name]
    if song_row.empty:
        print(f"Song '{song_name}' not found in the dataset.")
        return None
    else:
        ## GET THE SONG INDEX
        song_index = song_row.index[0]
        print(song_index)
        ## TRANSFORM THE SONG
        song_vector = transformed_data[song_index].reshape(1, -1)
        ## CALCULATE THE COSINE SIMILARITY
        similarity_scores = cosine_similarity(transformed_data, song_vector)
        print(similarity_scores.shape)
        ## GET THE TOP K SIMILAR SONGS INDEXES
        top_k_songs_indexes = np.argsort(similarity_scores.ravel())[-(k+1):][::-1]
        print(top_k_songs_indexes)
        ## GET THE TOP K SONGS NAMES
        top_k_songs_names = songs_data.loc[top_k_songs_indexes]
        ## PRINT THE TOP K SONGS NAMES
        top_k_list = top_k_songs_names[['name', 'artist','spotify_preview_url']].reset_index(drop=True)
        return top_k_list

In [70]:
## RECOMMEND SONG USING THE FUNCTION
recommend('Whenever, Wherever',songs_data=df_songs,transformed_data=transformed_df,k=10)

1025
(50674, 1)
[ 1025 12305  6046  6129 17241  6133  7172  6121  6526 38383  6287]


,name,artist,spotify_preview_url
0,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...
1,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...
2,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...
3,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...
4,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...
5,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...
6,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...
7,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...
8,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...
9,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...
